# Flickr30k Validation Report

Notebook báo cáo này **không đọc Held-out Test** và có thể chạy song song
với notebook Test một lần. Nó chỉ chuyển registry Validation đã khóa thành
bảng CSV/Markdown dùng cho báo cáo khoa học.

## Kaggle input bắt buộc — đúng 2 input

1. Output `F30K_Global_Val_Lock`;
2. Output `F30K_DenseMap_Closeout`.

Runtime: CPU, Accelerator None, Internet Off; dự kiến dưới 2 phút.


## Cell 1 — Xác minh khóa và xuất bảng báo cáo


In [1]:
import csv
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "F30K_Val_Report"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def unique_named(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (name, matches)
    return matches[0]

def write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

lock_path = unique_named("F30K_GLOBAL_VALIDATION_LOCK.json")
dense_path = unique_named("F30K_DenseMap_Closeout_RESULT.json")
lock = json.loads(lock_path.read_text(encoding="utf-8"))
dense = json.loads(dense_path.read_text(encoding="utf-8"))

assert lock["status"] == "locked"
assert lock["evaluation_boundary"] == "Validation-only method selection"
assert lock["test_set_used"] is False
assert dense["status"] == "complete"
assert dense["test_set_used"] is False
assert dense["retraining_authorized"] is False
assert dense["decision"] == "STOP_VALIDATION_NO_GAIN"

rows = lock["candidate_registry"]
assert len(rows) == 4
assert sum(row["decision"] == "PROMOTE_TO_FINAL_METHOD_LOCK" for row in rows) == 1
assert next(
    row for row in rows
    if row["decision"] == "PROMOTE_TO_FINAL_METHOD_LOCK"
)["pair_id"] == "T3S3_LogitKD"

csv_path = WORK_ROOT / "f30k_validation_method_summary_x100.csv"
columns = [
    "pair_id",
    "method",
    "baseline_CIDEr_D_mean_x100",
    "kd_CIDEr_D_mean_x100",
    "delta_CIDEr_D_mean_x100",
    "delta_CIDEr_D_sample_sd_x100",
    "combined_ci95_lower_x100",
    "combined_ci95_upper_x100",
    "decision",
]
with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)

report_path = WORK_ROOT / "F30K_VALIDATION_REPORT.md"
lines = [
    "# Flickr30k — khóa phương pháp trên Validation",
    "",
    "| Cặp | Phương pháp | Baseline CIDEr-D | KD CIDEr-D | Δ CIDEr-D | CI 95% | Quyết định |",
    "|---|---|---:|---:|---:|---:|---|",
]
for row in rows:
    lines.append(
        "| {pair_id} | {method} | {baseline:.2f} | {kd:.2f} | "
        "{delta:+.2f} | [{lower:.2f}; {upper:.2f}] | {decision} |".format(
            pair_id=row["pair_id"],
            method=row["method"],
            baseline=row["baseline_CIDEr_D_mean_x100"],
            kd=row["kd_CIDEr_D_mean_x100"],
            delta=row["delta_CIDEr_D_mean_x100"],
            lower=row["combined_ci95_lower_x100"],
            upper=row["combined_ci95_upper_x100"],
            decision=row["decision"],
        )
    )
lines += [
    "",
    "Tất cả metric trong bảng ở thang ×100 và làm tròn hai chữ số.",
    "Held-out Test không được notebook báo cáo này đọc.",
]
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

result = {
    "schema_version": "f30k_validation_report_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "source_lock_sha256": sha256(lock_path),
    "source_dense_closeout_sha256": sha256(dense_path),
    "candidate_count": len(rows),
    "selected_pair": lock["selected_method"]["pair_id"],
    "selected_method": lock["selected_method"]["method"],
    "held_out_test_read": False,
    "artifacts": {},
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
for path in [csv_path, report_path]:
    result["artifacts"][path.name] = {
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    }
write_json(result_path, result)
manifest_path = WORK_ROOT / "MANIFEST.json"
write_json(manifest_path, {
    "schema_version": "f30k_validation_report_manifest_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "artifacts": [
        {
            "relative_path": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in [csv_path, report_path, result_path]
    ],
})

print("Status                 : COMPLETE")
print("Candidates             :", len(rows))
print("Selected               :", result["selected_pair"], "/", result["selected_method"])
print("DenseMap decision      :", dense["decision"])
print("Held-out Test read     : False")
print("CSV                     :", csv_path)
print("Markdown                :", report_path)
print("Manifest                :", manifest_path)


Status                 : COMPLETE
Candidates             : 4
Selected               : T3S3_LogitKD / Direct Logit-KD
DenseMap decision      : STOP_VALIDATION_NO_GAIN
Held-out Test read     : False
CSV                     : /kaggle/working/F30K_Val_Report/f30k_validation_method_summary_x100.csv
Markdown                : /kaggle/working/F30K_Val_Report/F30K_VALIDATION_REPORT.md
Manifest                : /kaggle/working/F30K_Val_Report/MANIFEST.json
